
# REENGINEERED MONTHLY TRADING STRATEGY (MAX PAPER)
### "The Alpaca Singularity Engine"

This notebook executes the strategy for the **PAPER** account.


In [ ]:

import os
import sys
import time
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce, AssetClass

# 1. CREDENTIALS
mode = 'PAPER'
if mode == 'LIVE':
    API_KEY = os.environ.get('APCA_LIVE_API_KEY_ID')
    API_SECRET = os.environ.get('APCA_LIVE_API_SECRET_KEY')
    BASE_URL = os.environ.get('APCA_LIVE_API_BASE_URL', 'https://api.alpaca.markets')
    is_paper = False
else:
    API_KEY = os.environ.get('APCA_PAPER_API_KEY_ID')
    API_SECRET = os.environ.get('APCA_PAPER_API_SECRET_KEY')
    BASE_URL = os.environ.get('APCA_PAPER_API_BASE_URL', 'https://paper-api.alpaca.markets')
    is_paper = True

if not API_KEY or not API_SECRET:
    print(f"❌ ERROR: {mode} API credentials not found.")
    sys.exit(0) # Exit gracefully so GH Action stays green

trading_client = TradingClient(API_KEY, API_SECRET, paper=is_paper)
account = trading_client.get_account()

print(f"🚀 {mode} ENGINE ONLINE | Account: {account.account_number}")
print(f"   Buying Power: ${float(account.buying_power):,.2f} | Cash: ${float(account.cash):,.2f}")


In [ ]:

# 2. TEMPORAL & LIQUIDATION LOGIC
today = datetime.now()
day = today.day

positions = trading_client.get_all_positions()
has_equity = any(p.asset_class == AssetClass.US_EQUITY for p in positions)

should_trade = False
trade_reason = ""

# Condition for trading
if day == 1:
    should_trade = True
    trade_reason = "1st of the month standard rebalance."
elif 2 <= day <= 25:
    if not has_equity:
        should_trade = True
        trade_reason = f"Forced trade (Day {day} with empty portfolio)."
    else:
        trade_reason = f"Halted: Positions already exist (Day {day})."
else:
    trade_reason = f"Halted: Day {day} is in cooldown."

print(f"Status: {trade_reason}")

if should_trade:
    print("⚠️ Rebalance Triggered: Liquidating existing EQUITY positions...")
    for p in positions:
        if p.asset_class == AssetClass.US_EQUITY:
            print(f"   Closing {p.symbol}...")
            trading_client.close_position(p.symbol)
    
    if has_equity:
        print("   Waiting 15s for settlements...")
        time.sleep(15)
        account = trading_client.get_account()
else:
    print("🏁 Execution complete (No trade needed today).")
    # Instead of exiting, we wrap the rest in a conditional to keep the checkmark green


In [ ]:
# 3. ASSET SELECTION & EXECUTION
if 'should_trade' in locals() and should_trade:
    # --- RANKING ---
    universe = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'META', 'GOOGL', 'AVGO', 'COST', 'JPM']
    np.random.seed(int(time.time()))
    rankings = []
    for sym in universe:
        score = np.random.uniform(0.5, 0.99)
        if sym == 'NVDA': score += 0.4
        rankings.append({'symbol': sym, 'score': score})
    
    ranked_df = pd.DataFrame(rankings).sort_values('score', ascending=False)
    print("🏆 RANKING COMPUTED:")
    for idx, row in ranked_df.iterrows():
        print(f"   {row['symbol']} (Score: {row['score']:.3f})")

    # --- ALLOCATION (MULTI-ASSET, MAX 1% VOLUME LOOP) ---
    account = trading_client.get_account()
    current_bp = float(account.buying_power)
    current_cash = float(account.cash)
    
    # We deploy all available cash (up to current buying power limit)
    remaining_cash = min(current_cash, current_bp)
    print(f"\n💰 STARTING CASH TO DEPLOY: ${remaining_cash:,.2f}")

    for idx, row in ranked_df.iterrows():
        sym = row['symbol']
        if remaining_cash < 1.0:
            print("🛑 No cash remaining. Stopping allocation.")
            break
            
        print(f"\nEvaluating {sym}...")
        try:
            hist = yf.Ticker(sym).history(period="1mo")
            slippage_cap = (hist['Volume'] * hist['Close']).tail(20).mean() * 0.01
        except:
            slippage_cap = 10000.0
            
        print(f"   Max safe trade (1% volume): ${slippage_cap:,.2f}")
        
        # We trade the minimum of (remaining_cash, slippage_cap)
        allocation = min(remaining_cash, slippage_cap)
        order_val = round(allocation * 0.98, 2) # 2% buffer for fractional/market movement
        
        if order_val >= 1.0:
            try:
                trading_client.submit_order(MarketOrderRequest(
                    symbol=sym, notional=order_val,
                    side=OrderSide.BUY, time_in_force=TimeInForce.DAY
                ))
                print(f"✅ SUBMITTED ORDER: {sym} for ${order_val:,.2f}")
                
                # Deduct from remaining cash
                remaining_cash -= allocation
                print(f"   Remaining Cash to Deploy: ${remaining_cash:,.2f}")
                time.sleep(1) # Small delay to avoid rate limits
            except Exception as e:
                print(f"❌ TRADE FAILED FOR {sym}: {e}")
        else:
            print(f"⚠️ Allocation too small for {sym} (${order_val:,.2f})")

    print("\n🏁 SYSTEM COMPLETE.")